In [15]:
# Import des données traitées
from numpy import select
import pandas as pd
import duckdb
pd.set_option('display.max_columns', None)

chemin_fichier_order = r"F_orders.parquet"
chemin_fichier_product = r"D_products.parquet"
chemin_fichier_canal = r"d_facture_clean.parquet"
chemin_fichier_client = r"dim_client.parquet"

df_ML_customer=duckdb.query(f"""
    with order_complete as(
        SELECT
            o.*,
            CASE
                WHEN o.TotalDiscountGBP=0 THEN 1
                ELSE 0
            END AS check_promotion,
            p.ProductCategory_Propre,
            c.SalesChannel,
            cl.CustomerSegment,
        From read_parquet('{chemin_fichier_order}') as o
        LEFT JOIN read_parquet('{chemin_fichier_product}') as p
            ON p.StockCode = o.StockCode
        LEFT JOIN read_parquet('{chemin_fichier_canal}') as c
            ON c.Invoice=o.Invoice
        LEFT JOIN read_parquet('{chemin_fichier_client}') as cl
            ON cl.CustomerID=o.CustomerID_clean
    )                        

    select 
        CustomerID_clean,
        CustomerSegment,
        COUNT(DISTINCT(OrderLineID)) as total_lignes_commandes,
        SUM(TotalPriceGBP) as total_CA,
        SUM(TotalPriceGBP_WithDiscount) as total_CA_apres_remise,
        SUM(Quantity_clean) as Total_quantite,
        SUM(Quantity_clean)/COUNT(DISTINCT(Invoice)) as quantite_moyenne_par_facture,
        SUM(TotalPriceGBP_WithDiscount)/SUM(Quantity_clean) as panier_moyen,
        SUM(TotalDiscountGBP)/SUM(TotalPriceGBP) as remise_moyenne,
        SUM(check_promotion) as nb_achat_avec_promo,
        SUM(check_promotion)/COUNT(DISTINCT(OrderLineID)) as pourcent_achat_avec_promo,
        MODE(country_clean) AS pays_prefere,
        MODE(ProductCategory_Propre) AS categorie_preferee,
        MODE(SalesChannel) AS canal_prefere,
        MODE(PromotionCode_clean) AS codePromotion_prefere,
        -- 1. LA RÉCENCE (en jours)
        -- On calcule la différence entre la date d'achat la plus récente (MAX) et notre date de référence
        DATE_DIFF('day', MAX(InvoiceDate_clean), CAST('2012-01-01' AS DATE)) AS recence_jours,
        -- 2. LA FRÉQUENCE SUR 1 AN (Nombre de factures distinctes)
        -- On compte les 'Invoice' uniques, mais UNIQUEMENT si l'achat a eu lieu dans les 365 jours avant la référence
        COUNT(DISTINCT Invoice) FILTER (WHERE DATE_DIFF('day', InvoiceDate_clean, CAST('2012-01-01' AS DATE)) <= 365) AS nb_achats_annee
    From order_complete
    GROUP BY CustomerID_clean, CustomerSegment
""").df()

In [6]:
def explorer_dataset(dataset):
        print(f"Start {'-'*10} Pour ce dataset, la taille est de {dataset.shape} {'-'*10}")
        print('-'*30)
        display(dataset.head(3))
        print('-'*30)
        display(dataset.tail(3))
        print('-'*30)
        print(f"Présences de valeurs 'NaN'.")
        print(f"{dataset.isna().sum()}")
        display(dataset[dataset.isna().any(axis=1)])
        print('-'*30)
        print(f"Les doublons")
        print(f"{dataset.duplicated().sum()}")
        print('-'*30)
        print(f"Les types de données")
        print(f"{dataset.dtypes}")
        print('-'*30)
        print(f"Description")
        display(dataset.describe().T)
        print("FIN"+'-'*100)

In [16]:
explorer_dataset(df_ML_customer)

Start ---------- Pour ce dataset, la taille est de (5943, 17) ----------
------------------------------


,CustomerID_clean,CustomerSegment,total_lignes_commandes,total_CA,total_CA_apres_remise,Total_quantite,quantite_moyenne_par_facture,panier_moyen,remise_moyenne,nb_achat_avec_promo,pourcent_achat_avec_promo,pays_prefere,categorie_preferee,canal_prefere,codePromotion_prefere,recence_jours,nb_achats_annee
0,15048,VIP,121,2955.380011,2686.497541,239.0,47.800000,11.240575,0.090981,53.0,0.438017,United Kingdom,other,Online,NONE,88,4
1,13425,VIP,178,30742.790165,28635.252180,2327.0,332.428571,12.305652,0.068554,25.0,0.140449,United Kingdom,other,Phone,WELCOME10,131,3
2,17863,Regular,274,51072.200328,47276.682155,4920.0,410.000000,9.609082,0.074317,82.0,0.299270,United Kingdom,other,Phone,NONE,82,5


------------------------------


,CustomerID_clean,CustomerSegment,total_lignes_commandes,total_CA,total_CA_apres_remise,Total_quantite,quantite_moyenne_par_facture,panier_moyen,remise_moyenne,nb_achat_avec_promo,pourcent_achat_avec_promo,pays_prefere,categorie_preferee,canal_prefere,codePromotion_prefere,recence_jours,nb_achats_annee
5940,13290,Low value,2,30.48,28.194,2.0,1.0,14.097,0.075,1.0,0.5,United Kingdom,gifts,Phone,NONE,453,0
5941,13100,VIP,1,15.24,12.954,1.0,1.0,12.954,0.150,0.0,0.0,United Kingdom,gifts,Phone,XMAS,647,0
5942,16994,VIP,1,15.24,15.240,1.0,1.0,15.240,0.000,1.0,1.0,United Kingdom,gifts,Marketplace,NONE,649,0


------------------------------
Présences de valeurs 'NaN'.
CustomerID_clean                0
CustomerSegment                 0
total_lignes_commandes          0
total_CA                        0
total_CA_apres_remise           0
Total_quantite                  0
quantite_moyenne_par_facture    0
panier_moyen                    0
remise_moyenne                  0
nb_achat_avec_promo             0
pourcent_achat_avec_promo       0
pays_prefere                    0
categorie_preferee              0
canal_prefere                   0
codePromotion_prefere           0
recence_jours                   0
nb_achats_annee                 0
dtype: int64


,CustomerID_clean,CustomerSegment,total_lignes_commandes,total_CA,total_CA_apres_remise,Total_quantite,quantite_moyenne_par_facture,panier_moyen,remise_moyenne,nb_achat_avec_promo,pourcent_achat_avec_promo,pays_prefere,categorie_preferee,canal_prefere,codePromotion_prefere,recence_jours,nb_achats_annee


------------------------------
Les doublons
0
------------------------------
Les types de données
CustomerID_clean                    str
CustomerSegment                     str
total_lignes_commandes            int64
total_CA                        float64
total_CA_apres_remise           float64
Total_quantite                  float64
quantite_moyenne_par_facture    float64
panier_moyen                    float64
remise_moyenne                  float64
nb_achat_avec_promo             float64
pourcent_achat_avec_promo       float64
pays_prefere                        str
categorie_preferee                  str
canal_prefere                       str
codePromotion_prefere               str
recence_jours                     int64
nb_achats_annee                   int64
dtype: object
------------------------------
Description


,count,mean,std,min,25%,50%,75%,max
total_lignes_commandes,5943.0,179.601380,3298.938358,1.00,20.000000,53.000000,142.000000,2.529750e+05
total_CA,5943.0,24775.860975,258726.484456,1.01,2150.509975,5755.819992,16461.415150,1.822348e+07
total_CA_apres_remise,5943.0,22334.985811,234410.711011,1.01,1934.555722,5174.557998,14837.703384,1.651994e+07
Total_quantite,5943.0,2089.351338,23421.199661,1.00,184.000000,483.000000,1370.000000,1.670108e+06
quantite_moyenne_par_facture,5943.0,213.541862,1303.555676,1.00,79.000000,134.000000,221.125000,8.716700e+04
panier_moyen,5943.0,11.031162,2.845721,1.01,9.379558,10.795079,12.328893,3.237600e+01
remise_moyenne,5943.0,0.100254,0.050353,0.00,0.066447,0.100000,0.133333,2.045659e-01
nb_achat_avec_promo,5943.0,35.940434,667.379383,0.00,0.000000,3.000000,30.000000,5.111400e+04
pourcent_achat_avec_promo,5943.0,0.197936,0.277683,0.00,0.000000,0.066667,0.300000,1.000000e+00
recence_jours,5943.0,225.457681,211.882061,23.00,47.000000,118.000000,404.000000,7.610000e+02


FIN----------------------------------------------------------------------------------------------------
